In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [4]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [5]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [6]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your OS:\n   - macOS: download and install the `.pkg`\n   - Windows: download and install the `.msi`\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and start a local model with:\n   ```bash\n   ollama run llama3\n   ```\n\n   This will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\n3. To test that the local server is running, use:\n   ```bash\n   curl http://localhost:11434\n   ```\n\nIf you want to use it from Python, install the client with:\n```bash\npip install ollama\n```'

In [6]:
assistant.rag("How do I run Olama locally?")

'I don’t see any FAQ entry in the provided context about running Olama locally.'

In [7]:
assistant.rag("Who is the president of United states?")

'The context does not say who the president of the United States is.'

In [6]:
assistant.rag("Who am I?")

'You are automatically assigned a random display name when you set up your account, such as “Lucid Elbakyan.”\n\nIf you want to know your display name, click **“Edit Course Profile”**. Your **first field** is your nickname/displayed name.'

In [7]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—usually you can still join if the course is open for enrollment.\n\nTo confirm, you’ll want to check:\n- whether registration is still open\n- if there’s a waitlist\n- whether there are any prerequisites or deadlines\n- if the course has a cap on enrollment\n\nIf you want, I can help you draft a quick message to the instructor or course admin asking to join.'

In [8]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

# def search(query):
#     boost_dict = {"question": 3.0, "section": 0.5}
#     filter_dict = {"course": "llm-zoomcamp"}

#     return index.search(
#         query,
#         num_results=5,
#         boost_dict=boost_dict,
#         filter_dict=filter_dict
#     )

In [9]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties":{
            "query":{
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


# search_tool = {
#     "type": "function",
#     "name": "search",
#     "description": "Search the FAQ database for entries matching the given query.",
#     "parameters": {
#         "type": "object",
#         "properties": {
#             "query": {
#                 "type": "string",
#                 "description": "Search query text to look up in the course FAQ."
#             }
#         },
#         "required": ["query"],
#         "additionalProperties": False
#     }
# }

In [10]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

# response = openai_client.responses.create(
#     model="gpt-5.4-mini",
#     input=messages,
#     tools=[search_tool],
# )

# response.output

[ResponseFunctionToolCall(arguments='{"query":"join the course late discovered course enroll can I join after start FAQ"}', call_id='call_PimoXZoMbTHoNw8IOmOLSvJi', name='search', type='function_call', id='fc_012f93a397d9cb8d006a2f61cbf90081929d71812d49b9705a', namespace=None, status='completed')]

In [11]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

# import json

# call = response.output[0]
# args = json.loads(call.arguments)

# results = search(**args)
# result_json = json.dumps(results, indent=2)

In [12]:
messages.extend(response.output)
messages.append({
    "type":"function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

# messages.extend(response.output)

# messages.append({
#     "type": "function_call_output",
#     "call_id": call.call_id,
#     "output": result_json,
# })

In [13]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open.'

In [14]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(654, 33)

In [16]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [17]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [18]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return{
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

# def make_call(call):
#     args = json.loads(call.arguments)

#     if call.name == "search":
#         result = search(**args)

#     result_json = json.dumps(result, indent=2)

#     return {
#         "type": "function_call_output",
#         "call_id": call.call_id,
#         "output": result_json,
#     }

<b>
    Processing one response

Let's process a single model response. We append each output entry to the conversation, print any messages, and run any function calls. Function-call results get appended too.
</b>

In [19]:
question = "What is my first question?"

messages = [
    {"role":"developer","content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls=False

for item in response.output:
    if item.type == "function_call":
        print("function_call", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls= True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)


# question = "I just discovered the course. Can I join it?"

# messages = [
#     {"role": "developer", "content": instructions},
#     {"role": "user", "content": question},
# ]

# response = openai_client.responses.create(
#     model="gpt-5.4-mini",
#     input=messages,
#     tools=[search_tool],
# )

# messages.extend(response.output)
# has_function_calls = False

# for item in response.output:
#     if item.type == "function_call":
#         print("function_call:", item.name, item.arguments)
#         call_output = make_call(item)
#         messages.append(call_output)
#         has_function_calls = True

#     elif item.type == "message":
#         print("ASSISTANT:")
#         print(item.content[0].text)

function_call search {"query":"first question FAQ what is my first question"}
function_call search {"query":"first question course FAQ"}
function_call search {"query":"what is my first question student"}


<b>

The full agent loop...


We wrap this in a while loop. The loop keeps calling the model until it returns a response without any function calls. We also keep an iteration counter so we can see how many round-trips happened.

</b>

In [20]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls=False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output=make_call(item)
            messages.append(call_output)
            has_function_calls=True

        elif item.type=="message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it+1
    if has_function_calls==False:
        break

# it = 1

# while True:
#     print(f"iteration #{it}...")
#     has_function_calls = False

#     response = openai_client.responses.create(
#         model="gpt-5.4-mini",
#         input=messages,
#         tools=[search_tool],
#     )

#     messages.extend(response.output)

#     for item in response.output:
#         if item.type == "function_call":
#             print("function_call:", item.name, item.arguments)
#             call_output = make_call(item)
#             messages.append(call_output)
#             has_function_calls = True

#         elif item.type == "message":
#             print("ASSISTANT:")
#             print(item.content[0].text)

#     it = it + 1
#     if has_function_calls == False:
#         break

iteration #1...
ASSISTANT:
Your first question appears to be: **“What is my first question?”**

If you meant something else, tell me what you’re looking for and I’ll help. Are there other areas you want to explore?


<b>
Wrapping it in a function.



Let's wrap the loop in a function so we can reuse it. The function takes the instructions and the question as parameters, and returns the final answer.
</b>

In [21]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

    

# def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
#     messages = [
#         {"role": "developer", "content": instructions},
#         {"role": "user", "content": question}
#     ]

#     it = 1

#     while True:
#         print(f"iteration #{it}...")
#         has_function_calls = False

#         response = openai_client.responses.create(
#             model=model,
#             input=messages,
#             tools=[search_tool]
#         )

#         messages.extend(response.output)

#         for item in response.output:
#             if item.type == "function_call":
#                 print("function_call:", item.name, item.arguments)
#                 call_output = make_call(item)
#                 messages.append(call_output)
#                 has_function_calls = True

#             elif item.type == "message":
#                 print("ASSISTANT:")
#                 last_answer = item.content[0].text
#                 print(item.content[0].text)

#         it = it + 1
#         if has_function_calls == False:
#             break

#     return last_answer

In [22]:
agent_loop(instructions, "How do i run Olama locally?")
# agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama local run install start model serve local"}
function_call: search {"query":"Olama locally run Ollama local setup"}
function_call: search {"query":"Ollama FAQ run locally models"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - **macOS**: download from https://ollama.com/download and install the `.pkg`
   - **Windows**: download the `.msi` from the same page
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This downloads the model and opens a local chat session.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from Ollama.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```
   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - **macOS**: download from https://ollama.com/download and install the `.pkg`\n   - **Windows**: download the `.msi` from the same page\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and opens a local chat session.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from Ollama.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection issue, you can also start the server directly with:\n```bash\nollama serve\n```\nor, in a notebook/terminal backgrou

In [23]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered can I still join course enrollment FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course even if you just discovered it. You can start learning and submit homework while the submission form is open.

One important caveat: if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.

If you'd like, I can also explain how certificates and homework work in the course.


"Yes — you can still join the course even if you just discovered it. You can start learning and submit homework while the submission form is open.\n\nOne important caveat: if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.\n\nIf you'd like, I can also explain how certificates and homework work in the course."

In [24]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered late can I join course enrollment late join discovered course"}
iteration #2...
function_call: search {"query":"certificate live cohort project submission while accepting submissions self-paced no certificate register interest course accepted start learning submit homework form open"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. Also, certificates are only available if you finish with a live cohort, not in self-paced mode.

If you’d like, I can also help with what you need to do first after joining.


'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open. Also, certificates are only available if you finish with a live cohort, not in self-paced mode.\n\nIf you’d like, I can also help with what you need to do first after joining.'

In [36]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit definition"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a **chess opening** that starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** to try to distract Black’s d-pawn and gain control of the center. It’s called a “gambit” because White is offering a pawn, though Black often doesn’t fully keep it.

A very common continuation is:
- **1. d4 d5 2. c4**

Main ideas:
- fight for the center
- open lines for White’s pieces
- create long-term pressure on Black’s position

If you meant the **Netflix show/book** “The Queen’s Gambit,” I can explain that too. Want the chess version in more detail, or the show?


'The **Queen’s Gambit** is a **chess opening** that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** to try to distract Black’s d-pawn and gain control of the center. It’s called a “gambit” because White is offering a pawn, though Black often doesn’t fully keep it.\n\nA very common continuation is:\n- **1. d4 d5 2. c4**\n\nMain ideas:\n- fight for the center\n- open lines for White’s pieces\n- create long-term pressure on Black’s position\n\nIf you meant the **Netflix show/book** “The Queen’s Gambit,” I can explain that too. Want the chess version in more detail, or the show?'

In [25]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"Queen's Gambit chess opening course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen gambit,” so it looks like this may be off-topic or not covered in the course materials.

If you meant something course-related, feel free to rephrase it and I can check again. Is there anything else you’d like to explore?


'I couldn’t find any course FAQ entry about “queen gambit,” so it looks like this may be off-topic or not covered in the course materials.\n\nIf you meant something course-related, feel free to rephrase it and I can check again. Is there anything else you’d like to explore?'

## ToyAIKit

In [26]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [27]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
# agent_tools = Tools()
# agent_tools.add_tool(search, search_tool)

In [44]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question":3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )


# def search(query: str) -> dict[str, str]:
#     """
#     Search the FAQ database for entries matching the given query.
#     """
#     return index.search(
#         query,
#         num_results=5,
#         boost_dict={"question": 3.0, "section": 0.5},
#         filter_dict={"course": "llm-zoomcamp"}
#     )

In [45]:
agent_tools = Tools()
agent_tools.add_tool(search)

# agent_tools = Tools()
# agent_tools.add_tool(search)

In [46]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

<b>

The output is the same JSON schema we hand-wrote in the function calling lesson. ToyAIKit generated it from the docstring and the type hint.

Every modern agent framework does this same trick. It reads a typed Python function with a docstring and builds the schema from it. The OpenAI Agents SDK, PydanticAI, LangChain and Google ADK all work this way. You write the tool and the framework figures out how to describe it.

</b>

**The chat interface and runner**

In [47]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools = agent_tools,
    developer_prompt = instructions,
    chat_interface=chat_interface,
    llm_client = OpenAIClient(model="gpt-5.4-mini")
)


# chat_interface = IPythonChatInterface()
# callback = DisplayingRunnerCallback(chat_interface)

# runner = OpenAIResponsesRunner(
#     tools=agent_tools,
#     developer_prompt=instructions,
#     chat_interface=chat_interface,
#     llm_client=OpenAIClient(model="gpt-5.4-mini")
# )

In [48]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

# result = runner.loop(
#     prompt="How do I run Olama locally?",
#     callback=callback,
# )

-> Response received


-> Response received


-> Response received


In [49]:
result.cost

CostInfo(input_cost=Decimal('0.00252075'), output_cost=Decimal('0.001161'), total_cost=Decimal('0.00368175'))

In [50]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local installation setup FAQ"}', call_id='ca

**Continuing the conversation**

**Take the messages from the previous result and pass them as previous_messages on the next loop call:**

In [55]:
result2 = runner.loop(
    prompt = "How do I run a different model?",
    previous_messages = result.all_messages,
    callback = callback,
)

# result2 = runner.loop(
#     prompt="How do I run a different model?",
#     previous_messages=result.all_messages,
#     callback=callback,
# )

-> Response received


-> Response received


In [57]:
result2 = runner.loop(
    prompt = "What is my second question?",
    previous_messages = result.all_messages,
    callback = callback,
)

-> Response received


In [58]:
runner.run()

You: what is my name?


-> Response received


-> Response received


You: who are you?


-> Response received


...

You: it is us political status.


-> Response received


You: stop


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='what is my name?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"name student what is my name course FAQ"}', call_id